<a href="https://www.kaggle.com/code/georgiyylp/module1-rus?scriptVersionId=290537536" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
import os

# Не даём transformers тянуть TensorFlow и Flax
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TRANSFORMERS_NO_FLAX"] = "1"

import torch
from torch import nn
from transformers import AutoModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

device: cuda


In [2]:
# ============= ЯЧЕЙКА 3: ДОПОЛНИТЕЛЬНЫЕ ИМПОРТЫ =============

# Для обработки аудио
import librosa
import librosa.display

# Для визуализации
import matplotlib.pyplot as plt
import seaborn as sns

# Для обработки сигналов
from scipy import signal

# Для ML метрик
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_curve, roc_auc_score, auc
)

# Для прогресс-баров в Kaggle
from tqdm.notebook import tqdm as tqdm_notebook

# Для работы с датами
from datetime import datetime

# PyTorch компоненты
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim

print("✅ Все дополнительные библиотеки импортированы!")


✅ Все дополнительные библиотеки импортированы!


In [3]:
# ============= НАСТРОЙКА ОКРУЖЕНИЯ KAGGLE =============

import os
import sys
import numpy as np
import pandas as pd
import torch

# Kaggle paths
PROJECT_DIR = '/kaggle/working'
DATA_DIR = f'{PROJECT_DIR}/data'
MODELS_DIR = f'{PROJECT_DIR}/models'
RESULTS_DIR = f'{PROJECT_DIR}/results'

# Создать структуру директорий
os.makedirs(f'{DATA_DIR}/raw', exist_ok=True)
os.makedirs(f'{DATA_DIR}/processed', exist_ok=True)
os.makedirs(f'{MODELS_DIR}/checkpoints', exist_ok=True)
os.makedirs(f'{RESULTS_DIR}/metrics', exist_ok=True)
os.makedirs(f'{RESULTS_DIR}/plots', exist_ok=True)

print("="*70)
print("🚀 НАСТРОЙКА KAGGLE ОКРУЖЕНИЯ")
print("="*70)
print(f"✅ Структура директорий создана")
print(f"   PROJECT: {PROJECT_DIR}")
print(f"   DATA:    {DATA_DIR}")
print(f"   MODELS:  {MODELS_DIR}")
print(f"   RESULTS: {RESULTS_DIR}")

# Проверить GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\n🔧 GPU Information:")
print(f"   GPU доступен: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   Устройство: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"   CUDA версия: {torch.version.cuda}")
else:
    print(f"   ⚠️ GPU не найден!")

# Проверить доступные датасеты
print(f"\n📂 Доступные Input датасеты:")
if os.path.exists('/kaggle/input'):
    datasets = os.listdir('/kaggle/input')
    if datasets:
        for dataset in datasets:
            print(f"   ✅ {dataset}")
    else:
        print(f"   ❌ Нет подключенных датасетов")
        print(f"   💡 Добавьте датасет: Add Data → Search 'asvspoof 2019'")
else:
    print(f"   ❌ /kaggle/input не существует")

print("="*70)

🚀 НАСТРОЙКА KAGGLE ОКРУЖЕНИЯ
✅ Структура директорий создана
   PROJECT: /kaggle/working
   DATA:    /kaggle/working/data
   MODELS:  /kaggle/working/models
   RESULTS: /kaggle/working/results

🔧 GPU Information:
   GPU доступен: True
   Устройство: Tesla T4
   VRAM: 15.8 GB
   CUDA версия: 12.6

📂 Доступные Input датасеты:
   ✅ deepfakedetector-ver1-0
   ✅ pyara


In [4]:
# ============= ЯЧЕЙКА 3: ПОДКЛЮЧЕНИЕ PYARA ДАТАСЕТА =============

"""
ИНСТРУКЦИЯ:
1. Откройте Settings этого notebook (⚙️ сверху)
2. В левой колонке найдите "Add Data" 
3. Нажмите "+ Add Data" → "Search"
4. Введите "pyara" → Найдите "alep079/pyara"
5. Нажмите "Add" → Датасет подключится автоматически
6. Перезапустите ячейку ниже
"""

import os

# ============= НАСТРОЙКА ПУТЕЙ =============

# ✅ PyAra на Kaggle
DATASET_NAME = 'pyara'
raw_dir = '/kaggle/input/pyara'

# Проверить существование датасета
print("=" * 70)
print("🔍 ПРОВЕРКА ДАТАСЕТА PYARA")
print("=" * 70)

if os.path.exists(raw_dir):
    print(f"✅ Датасет найден: {raw_dir}")
    print(f"\n📂 Структура директорий:")
    
    # Показать структуру (первые 3 уровня)
    for root, dirs, files in os.walk(raw_dir):
        level = root.replace(raw_dir, '').count(os.sep)
        indent = ' ' * 2 * level
        
        # Показываем папку
        folder_name = os.path.basename(root) if root != raw_dir else 'pyara'
        print(f'{indent}📁 {folder_name}/')
        
        # Показываем первые 3 файла в папке
        if level < 2:  # Только на первых двух уровнях
            subindent = ' ' * 2 * (level + 1)
            for file in files[:3]:
                print(f'{subindent}📄 {file}')
            if len(files) > 3:
                print(f'{subindent}... и ещё {len(files) - 3} файлов')
            
            # Показываем подпапки (но не их содержимое)
            for dir_name in dirs[:3]:
                print(f'{subindent}📁 {dir_name}/')
            if len(dirs) > 3:
                print(f'{subindent}... и ещё {len(dirs) - 3} папок')

else:
    print(f"❌ ОШИБКА: Датасет НЕ найден по пути {raw_dir}")
    print(f"\n💡 РЕШЕНИЕ:")
    print(f"   1. Откройте Settings (⚙️ в левом меню)")
    print(f"   2. Нажмите '+ Add Data' → Search 'pyara'")
    print(f"   3. Выберите 'alep079/pyara'")
    print(f"   4. Нажмите 'Add'")
    print(f"   5. Перезапустите эту ячейку")
    print(f"\n⚠️ БЕЗ ДАТАСЕТА ДАЛЬШЕ НЕВОЗМОЖНО ПРОДОЛЖИТЬ!")
    raise FileNotFoundError(f"Dataset not found at {raw_dir}")

print("\n" + "=" * 70)
print("✅ ДАТАСЕТ УСПЕШНО ПОДКЛЮЧЕН!")
print("=" * 70)


🔍 ПРОВЕРКА ДАТАСЕТА PYARA
✅ Датасет найден: /kaggle/input/pyara

📂 Структура директорий:
📁 pyara/
  📁 final_dataset/
  📁 final_dataset/
    📄 final_dataset.tsv
    📁 Fake/
    📁 Real/
    📁 Fake/
    📁 Real/

✅ ДАТАСЕТ УСПЕШНО ПОДКЛЮЧЕН!


In [5]:
# ============= УЛУЧШЕННАЯ ЯЧЕЙКА 4: С ПРОГРЕССБАРОМ =============

"""
ПРАВИЛЬНАЯ СТРУКТУРА PYARA НА KAGGLE:
/kaggle/input/pyara/
├── final_dataset/
│   ├── Fake/       ← .wav файлы напрямую (без подпапок)
│   ├── Real/       ← .wav файлы напрямую (без подпапок)
│   └── final_dataset.tsv
"""

from tqdm.notebook import tqdm

def prepare_pyara_metadata(raw_dir: str, output_csv: str) -> pd.DataFrame:
    """Подготовка метаданных для PyAra датасета с прогрессбаром"""
    
    print("=" * 70)
    print("📊 ПОДГОТОВКА МЕТАДАННЫХ PYARA")
    print("=" * 70)
    
    # ===== ПРАВИЛЬНЫЕ ПУТИ =====
    # Реальная структура: /kaggle/input/pyara/final_dataset/Real и Fake
    dataset_dir = os.path.join(raw_dir, 'final_dataset')
    real_dir = os.path.join(dataset_dir, 'Real')
    fake_dir = os.path.join(dataset_dir, 'Fake')
    
    print(f"\n📂 Основные директории:")
    print(f"   Dataset root: {dataset_dir} -> {'✅' if os.path.exists(dataset_dir) else '❌'}")
    print(f"   Real (bonafide): {real_dir} -> {'✅' if os.path.exists(real_dir) else '❌'}")
    print(f"   Fake (spoofed):  {fake_dir}  -> {'✅' if os.path.exists(fake_dir) else '❌'}")
    
    # ===== ШАГ 1: ПОДСЧИТАТЬ ФАЙЛЫ (для прогрессбара) =====
    print(f"\n🔍 Подсчёт файлов перед обработкой...")
    
    real_files = []
    fake_files = []
    
    if os.path.isdir(real_dir):
        real_files = [f for f in os.listdir(real_dir) 
                     if f.endswith(('.wav', '.mp3', '.flac', '.ogg'))]
    
    if os.path.isdir(fake_dir):
        fake_files = [f for f in os.listdir(fake_dir) 
                     if f.endswith(('.wav', '.mp3', '.flac', '.ogg'))]
    
    total_files = len(real_files) + len(fake_files)
    print(f"   Real файлы: {len(real_files)}")
    print(f"   Fake файлы: {len(fake_files)}")
    print(f"   Всего: {total_files}")
    
    if total_files == 0:
        print(f"\n❌ КРИТИЧЕСКАЯ ОШИБКА: Не найдено ни одного аудиофайла!")
        return None
    
    data_list = []
    
    # ===== КЛАСС 0: Real (Bonafide) =====
    print(f"\n🔍 Обработка Real (bonafide) файлов...")
    
    for audio_file in tqdm(real_files, desc="Real files", unit="file"):
        audio_path = os.path.join(real_dir, audio_file)
        
        # Получить длительность
        try:
            y, sr = librosa.load(audio_path, sr=16000)
            duration = len(y) / sr
        except Exception as e:
            if len([d for d in data_list if d['label'] == 0]) < 5:
                print(f"   ⚠️ Ошибка при чтении {audio_file}: {e}")
            duration = 3.0  # Default
        
        data_list.append({
            'filename': audio_file,
            'label': 0,  # 0 = Real (bonafide)
            'duration': duration,
            'class_name': 'real',
            'full_path': audio_path
        })
    
    print(f"   ✅ Обработано {len(real_files)} Real файлов")
    
    # ===== КЛАСС 1: Fake (Spoofed) =====
    print(f"\n🔍 Обработка Fake (spoofed) файлов...")
    
    for audio_file in tqdm(fake_files, desc="Fake files", unit="file"):
        audio_path = os.path.join(fake_dir, audio_file)
        
        # Получить длительность
        try:
            y, sr = librosa.load(audio_path, sr=16000)
            duration = len(y) / sr
        except Exception as e:
            if len([d for d in data_list if d['label'] == 1]) < 5:
                print(f"   ⚠️ Ошибка при чтении {audio_file}: {e}")
            duration = 3.0
        
        data_list.append({
            'filename': audio_file,
            'label': 1,  # 1 = Fake (spoofed)
            'duration': duration,
            'class_name': 'fake',
            'full_path': audio_path
        })
    
    print(f"   ✅ Обработано {len(fake_files)} Fake файлов")
    
    # ===== СОЗДАТЬ DATAFRAME =====
    if len(data_list) == 0:
        print(f"\n❌ КРИТИЧЕСКАЯ ОШИБКА: Не найдено ни одного аудиофайла!")
        return None
    
    df = pd.DataFrame(data_list)
    
    print(f"\n📊 СТАТИСТИКА ДАТАСЕТА:")
    print(f"   Всего файлов: {len(df)}")
    
    # Баланс классов
    bonafide_count = (df['label'] == 0).sum()
    fake_count = (df['label'] == 1).sum()
    
    print(f"\n   Распределение по классам:")
    print(f"   - Real (bonafide): {bonafide_count} ({bonafide_count/len(df)*100:.1f}%)")
    print(f"   - Fake (spoofed):  {fake_count} ({fake_count/len(df)*100:.1f}%)")
    
    if fake_count == 0:
        print(f"\n❌ ОШИБКА: Нет Fake файлов!")
        return None
    
    # Средняя длительность
    print(f"\n   📏 Длительность аудио:")
    print(f"   - Min:  {df['duration'].min():.2f} сек")
    print(f"   - Max:  {df['duration'].max():.2f} сек")
    print(f"   - Mean: {df['duration'].mean():.2f} сек")
    
    # Сохранить
    os.makedirs(os.path.dirname(output_csv), exist_ok=True)
    df.to_csv(output_csv, index=False)
    print(f"\n✅ Метаданные сохранены: {output_csv}")
    
    return df


# ===== ВЫЗОВ =====
metadata_csv = f"{DATA_DIR}/processed/pyara_metadata.csv"
df_metadata = prepare_pyara_metadata(raw_dir, metadata_csv)

if df_metadata is None:
    print(f"\n❌ ОШИБКА: Не удалось подготовить метаданные!")
    raise RuntimeError("Failed to prepare metadata")

print(f"\n✅ УСПЕШНО: Датасет подготовлен и готов к использованию!")

📊 ПОДГОТОВКА МЕТАДАННЫХ PYARA

📂 Основные директории:
   Dataset root: /kaggle/input/pyara/final_dataset -> ✅
   Real (bonafide): /kaggle/input/pyara/final_dataset/Real -> ✅
   Fake (spoofed):  /kaggle/input/pyara/final_dataset/Fake  -> ✅

🔍 Подсчёт файлов перед обработкой...
   Real файлы: 73583
   Fake файлы: 128195
   Всего: 201778

🔍 Обработка Real (bonafide) файлов...


Real files:   0%|          | 0/73583 [00:00<?, ?file/s]

   ✅ Обработано 73583 Real файлов

🔍 Обработка Fake (spoofed) файлов...


Fake files:   0%|          | 0/128195 [00:00<?, ?file/s]

   ✅ Обработано 128195 Fake файлов

📊 СТАТИСТИКА ДАТАСЕТА:
   Всего файлов: 201778

   Распределение по классам:
   - Real (bonafide): 73583 (36.5%)
   - Fake (spoofed):  128195 (63.5%)

   📏 Длительность аудио:
   - Min:  2.99 сек
   - Max:  10.00 сек
   - Mean: 5.35 сек

✅ Метаданные сохранены: /kaggle/working/data/processed/pyara_metadata.csv

✅ УСПЕШНО: Датасет подготовлен и готов к использованию!


In [6]:
# ============= ЯЧЕЙКА 5: НОРМАЛИЗАЦИЯ И ФИЛЬТРАЦИЯ =============

"""
PyAra имеет переменную длительность аудио.
Нам нужно выбрать целевую длину и отфильтровать слишком короткие файлы.
"""

# ===== ВЫБОР ЦЕЛЕВОЙ ДЛИНЫ =====
TARGET_DURATION = 3.0  # Секунды (стандарт для ASVspoof, WavLM)
MIN_DURATION = 0.5     # Минимальная длина (фильтруем очень короткие)
MAX_DURATION = 10.0    # Максимальная длина (опционально)

print("=" * 70)
print("🎯 НОРМАЛИЗАЦИЯ И ФИЛЬТРАЦИЯ ПО ДЛИТЕЛЬНОСТИ")
print("=" * 70)

print(f"\n⚙️ Параметры фильтрации:")
print(f"   Target Duration: {TARGET_DURATION} сек")
print(f"   Min Duration:    {MIN_DURATION} сек")
print(f"   Max Duration:    {MAX_DURATION} сек (unlimited)")

# ===== ДО ФИЛЬТРАЦИИ =====
print(f"\n📊 ДО фильтрации:")
print(f"   Всего файлов: {len(df_metadata)}")
print(f"   Real (bonafide): {(df_metadata['label'] == 0).sum()}")
print(f"   Fake (spoofed):  {(df_metadata['label'] == 1).sum()}")

# ===== ФИЛЬТРАЦИЯ =====
df_metadata_filtered = df_metadata[
    (df_metadata['duration'] >= MIN_DURATION) &
    (df_metadata['duration'] <= MAX_DURATION)
].copy()

print(f"\n🔍 ПОСЛЕ фильтрации (длительность {MIN_DURATION}-{MAX_DURATION} сек):")
print(f"   Всего файлов: {len(df_metadata_filtered)}")
print(f"   Real (bonafide): {(df_metadata_filtered['label'] == 0).sum()}")
print(f"   Fake (spoofed):  {(df_metadata_filtered['label'] == 1).sum()}")

# Сколько удалили
removed = len(df_metadata) - len(df_metadata_filtered)
if removed > 0:
    print(f"   ⚠️ Удалено файлов: {removed} ({removed/len(df_metadata)*100:.1f}%)")
else:
    print(f"   ✅ Все файлы прошли фильтрацию")

# ===== ОБНОВИТЬ DATAFRAME =====
df_metadata = df_metadata_filtered.reset_index(drop=True)

# Добавить информацию о целевых сэмплах
df_metadata['target_samples'] = int(TARGET_DURATION * 16000)

# Сохранить обновленные метаданные
df_metadata.to_csv(metadata_csv, index=False)

print(f"\n✅ Отфильтрованные метаданные сохранены")
print(f"\n📊 Итоговая статистика:")
print(f"   Файлов: {len(df_metadata)}")
print(f"   Real: {(df_metadata['label'] == 0).sum()}")
print(f"   Fake: {(df_metadata['label'] == 1).sum()}")


🎯 НОРМАЛИЗАЦИЯ И ФИЛЬТРАЦИЯ ПО ДЛИТЕЛЬНОСТИ

⚙️ Параметры фильтрации:
   Target Duration: 3.0 сек
   Min Duration:    0.5 сек
   Max Duration:    10.0 сек (unlimited)

📊 ДО фильтрации:
   Всего файлов: 201778
   Real (bonafide): 73583
   Fake (spoofed):  128195

🔍 ПОСЛЕ фильтрации (длительность 0.5-10.0 сек):
   Всего файлов: 201778
   Real (bonafide): 73583
   Fake (spoofed):  128195
   ✅ Все файлы прошли фильтрацию

✅ Отфильтрованные метаданные сохранены

📊 Итоговая статистика:
   Файлов: 201778
   Real: 73583
   Fake: 128195


In [7]:
# ============= ЯЧЕЙКА 6: TRAIN/VAL/TEST SPLIT =============

"""
ЭТА ЯЧЕЙКА ОСТАЁТСЯ ТОЙ ЖЕ, что в твоём текущем notebook!
Просто скопируйте её из вашего diploma.ipynb (ячейка 5)

Ниже я приведу готовый код для полноты.
"""
from sklearn.model_selection import train_test_split


def create_train_val_test_split(df: pd.DataFrame,
                                train_size: float = 0.7,
                                val_size: float = 0.15,
                                test_size: float = 0.15,
                                random_state: int = 42) -> tuple:
    """Разделяет датасет на train/val/test с сохранением баланса"""
    
    print("\n" + "=" * 70)
    print("🔀 РАЗДЕЛЕНИЕ ДАТАСЕТА НА TRAIN/VAL/TEST")
    print("=" * 70)
    
    print("🔀 Разделение датасета на train/val/test...")
    
    total = train_size + val_size + test_size
    assert abs(total - 1.0) < 1e-6, f"Размеры должны суммироваться в 1.0, получено {total}"
    
    # Сначала разделить на train и (val+test)
    df_train, df_rest = train_test_split(
        df,
        test_size=(val_size + test_size),
        random_state=random_state,
        stratify=df['label']
    )
    
    # Потом разделить rest на val и test
    val_test_ratio = val_size / (val_size + test_size)
    df_val, df_test = train_test_split(
        df_rest,
        test_size=(1 - val_test_ratio),
        random_state=random_state,
        stratify=df_rest['label']
    )
    
    print(f"\n📊 Размеры наборов:")
    print(f"   Train: {len(df_train):4d} файлов ({len(df_train)/len(df)*100:5.1f}%)")
    print(f"   Val:   {len(df_val):4d} файлов ({len(df_val)/len(df)*100:5.1f}%)")
    print(f"   Test:  {len(df_test):4d} файлов ({len(df_test)/len(df)*100:5.1f}%)")
    
    # КРИТИЧНО: Перемешать КАЖДЫЙ набор отдельно
    df_train = df_train.sample(frac=1, random_state=42).reset_index(drop=True)
    df_val = df_val.sample(frac=1, random_state=42).reset_index(drop=True)
    df_test = df_test.sample(frac=1, random_state=42).reset_index(drop=True)
    print(f"\n✅ Каждый набор перемешан (shuffle)")
    
    # Проверить баланс
    print(f"\n📊 Баланс классов:")
    
    def print_class_balance(name, split_df):
        bonafide = (split_df['label'] == 0).sum()
        spoofed = (split_df['label'] == 1).sum()
        total = len(split_df)
        ratio = bonafide / max(spoofed, 1)
        
        print(f"   {name}:")
        print(f"      Real (bonafide): {bonafide:5d} ({bonafide/total*100:5.1f}%)")
        print(f"      Fake (spoofed):  {spoofed:5d} ({spoofed/total*100:5.1f}%)")
        print(f"      Ratio: {ratio:.2f}:1")
        
        return bonafide, spoofed
    
    print_class_balance("Train", df_train)
    print_class_balance("Val", df_val)
    print_class_balance("Test", df_test)
    
    # Вычислить веса для балансировки классов
    print(f"\n⚖️ ВЫЧИСЛЕНИЕ ВЕСОВ КЛАССОВ:")
    
    n_bonafide = (df_train['label'] == 0).sum()
    n_spoofed = (df_train['label'] == 1).sum()
    total_train = len(df_train)
    
    if n_spoofed > 0:
        weight_bonafide = total_train / (2 * n_bonafide)
        weight_spoofed = total_train / (2 * n_spoofed)
        
        print(f"   Real (bonafide) weight: {weight_bonafide:.4f}")
        print(f"   Fake (spoofed) weight:  {weight_spoofed:.4f}")
        
        class_weights = {
            'bonafide': weight_bonafide,
            'spoofed': weight_spoofed
        }
        print(f"\n✅ Веса вычислены")
    else:
        print(f"   ⚠️ Нет spoofed примеров!")
        class_weights = None
    
    return df_train, df_val, df_test, class_weights


# ===== ВЫЗОВ =====
df_train, df_val, df_test, class_weights = create_train_val_test_split(df_metadata)

FRACTION = 0.7  # Используем только 30% датасета

print(f"\n⚡ ОПТИМИЗАЦИЯ: Используем только {FRACTION*100:.0f}% датасета для ускорения")
print(f"   Было файлов: Train={len(df_train)}, Val={len(df_val)}, Test={len(df_test)}")

# Случайная выборка с сохранением баланса классов
df_train = df_train.groupby('label', group_keys=False).apply(
    lambda x: x.sample(frac=FRACTION, random_state=42)
).reset_index(drop=True)

df_val = df_val.groupby('label', group_keys=False).apply(
    lambda x: x.sample(frac=FRACTION, random_state=42)
).reset_index(drop=True)

df_test = df_test.groupby('label', group_keys=False).apply(
    lambda x: x.sample(frac=FRACTION, random_state=42)
).reset_index(drop=True)

print(f"   Стало файлов: Train={len(df_train)}, Val={len(df_val)}, Test={len(df_test)}")
print(f"   ✅ Ускорение в ~{1/FRACTION:.1f}x раз!")

# Сохранить split'ы
df_train.to_csv(f'{DATA_DIR}/processed/train_split.csv', index=False)
df_val.to_csv(f'{DATA_DIR}/processed/val_split.csv', index=False)
df_test.to_csv(f'{DATA_DIR}/processed/test_split.csv', index=False)

if class_weights:
    import json
    with open(f'{DATA_DIR}/processed/class_weights.json', 'w') as f:
        json.dump(class_weights, f, indent=2)
    print(f"✅ Веса сохранены")

print(f"\n✅ Split'ы созданы и сохранены в {DATA_DIR}/processed/")



🔀 РАЗДЕЛЕНИЕ ДАТАСЕТА НА TRAIN/VAL/TEST
🔀 Разделение датасета на train/val/test...

📊 Размеры наборов:
   Train: 141244 файлов ( 70.0%)
   Val:   30267 файлов ( 15.0%)
   Test:  30267 файлов ( 15.0%)

✅ Каждый набор перемешан (shuffle)

📊 Баланс классов:
   Train:
      Real (bonafide): 51508 ( 36.5%)
      Fake (spoofed):  89736 ( 63.5%)
      Ratio: 0.57:1
   Val:
      Real (bonafide): 11037 ( 36.5%)
      Fake (spoofed):  19230 ( 63.5%)
      Ratio: 0.57:1
   Test:
      Real (bonafide): 11038 ( 36.5%)
      Fake (spoofed):  19229 ( 63.5%)
      Ratio: 0.57:1

⚖️ ВЫЧИСЛЕНИЕ ВЕСОВ КЛАССОВ:
   Real (bonafide) weight: 1.3711
   Fake (spoofed) weight:  0.7870

✅ Веса вычислены

⚡ ОПТИМИЗАЦИЯ: Используем только 70% датасета для ускорения
   Было файлов: Train=141244, Val=30267, Test=30267
   Стало файлов: Train=98871, Val=21187, Test=21187
   ✅ Ускорение в ~1.4x раз!


/tmp/ipykernel_55/2218866331.py:111: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_train = df_train.groupby('label', group_keys=False).apply(
/tmp/ipykernel_55/2218866331.py:115: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_val = df_val.groupby('label', group_keys=False).apply(
/tmp/ipykernel_55/2218866331.py:119: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behav

✅ Веса сохранены

✅ Split'ы созданы и сохранены в /kaggle/working/data/processed/


In [8]:
"""
ПРЕПРОЦЕССИНГ АУДИО

Нормализация → Фильтр высоких частот → Pad/Trim до целевой длины
"""

class AudioPreprocessor:
    """Препроцессор для аудиофайлов"""

    def __init__(self, sr: int = 16000, duration: float = 3.0):
        """
        sr: sampling rate (16 kHz - стандарт для ASVspoof)
        duration: целевая длительность в секундах
        """
        self.sr = sr
        self.duration = duration
        self.target_samples = int(sr * duration)
        print(f"✅ AudioPreprocessor инициализирован")
        print(f"   SR: {sr} Hz, Duration: {duration} sec, Samples: {self.target_samples}")

    def load_audio(self, file_path: str) -> np.ndarray:
        """Загружает аудиофайл"""
        try:
            y, sr = librosa.load(file_path, sr=self.sr, mono=True)
            return y
        except Exception as e:
            print(f"❌ Ошибка загрузки {os.path.basename(file_path)}: {e}")
            return None

    def normalize(self, audio: np.ndarray) -> np.ndarray:
        """Нормализует амплитуду аудио к [-1, 1]"""
        max_val = np.max(np.abs(audio))
        if max_val > 0:
            audio = audio / (max_val + 1e-7)
        return audio

    def high_pass_filter(self, audio: np.ndarray, cutoff_hz: float = 200) -> np.ndarray:
        """Применяет фильтр высоких частот (убирает низкочастотный шум)"""
        from scipy import signal
        sos = signal.butter(5, cutoff_hz, 'hp', fs=self.sr, output='sos')
        audio = signal.sosfilt(sos, audio)
        return audio

    def pad_or_trim(self, audio: np.ndarray) -> np.ndarray:
        """Обрезает или паддирует аудио до целевой длины"""
        if len(audio) > self.target_samples:
            # Если длиннее - обрезать с начала
            audio = audio[:self.target_samples]
        else:
            # Если короче - паддировать нулями в конце
            audio = np.pad(audio, (0, self.target_samples - len(audio)), mode='constant')
        return audio

    def preprocess(self, file_path: str) -> np.ndarray:
        """Полный препроцессинг: Load → Normalize → HPF → Pad/Trim"""

        # 1. Загрузить
        audio = self.load_audio(file_path)
        if audio is None:
            return None

        # 2. Нормализовать
        audio = self.normalize(audio)

        # 3. Фильтр высоких частот
        audio = self.high_pass_filter(audio)

        # 4. Pad/trim до целевой длины
        audio = self.pad_or_trim(audio)

        return audio

# Инициализировать препроцессор
print("="*70)
print("🔧 ИНИЦИАЛИЗАЦИЯ AUDIO PREPROCESSOR")
print("="*70)

preprocessor = AudioPreprocessor(sr=16000, duration=3.0)

# Тестировать на примере
if df_train is not None and len(df_train) > 0:
    print("\n🧪 Тестирование препроцессора на примере...")

    # Найти аудио директорию
    audio_dir = None
    if os.path.exists(f'{raw_dir}/LA/ASVspoof2019_LA_dev/flac'):
        audio_dir = f'{raw_dir}/LA/ASVspoof2019_LA_dev/flac'
    elif os.path.exists(f'{raw_dir}/ASVspoof2019_LA_dev/flac'):
        audio_dir = f'{raw_dir}/ASVspoof2019_LA_dev/flac'

    if audio_dir:
        sample_filename = df_train.iloc[0]['filename']
        sample_file = os.path.join(audio_dir, sample_filename)

        if os.path.exists(sample_file):
            audio_processed = preprocessor.preprocess(sample_file)

            if audio_processed is not None:
                print(f"✅ Препроцессинг работает!")
                print(f"   Файл: {sample_filename}")
                print(f"   Форма аудио: {audio_processed.shape}")
                print(f"   Длительность: {len(audio_processed) / 16000:.2f} сек")
                print(f"   Min/Max значения: [{audio_processed.min():.3f}, {audio_processed.max():.3f}]")

                # Визуализация
                fig, axes = plt.subplots(2, 1, figsize=(12, 6))

                # Исходное аудио
                y_raw, sr = librosa.load(sample_file, sr=16000)
                axes[0].plot(y_raw[:16000*3], linewidth=0.5, alpha=0.7)
                axes[0].set_title(f'Исходное аудио: {sample_filename}')
                axes[0].set_ylabel('Амплитуда')
                axes[0].grid(True, alpha=0.3)

                # Обработанное аудио
                axes[1].plot(audio_processed, linewidth=0.5, alpha=0.7, color='orange')
                axes[1].set_title('Обработанное аудио (Нормализовано + HPF + Pad)')
                axes[1].set_ylabel('Амплитуда')
                axes[1].set_xlabel('Сэмпл')
                axes[1].grid(True, alpha=0.3)

                plt.tight_layout()
                plt.savefig(f'{RESULTS_DIR}/plots/preprocessing_example.png', dpi=100, bbox_inches='tight')
                plt.show()

                print(f"✅ График сохранён!")

🔧 ИНИЦИАЛИЗАЦИЯ AUDIO PREPROCESSOR
✅ AudioPreprocessor инициализирован
   SR: 16000 Hz, Duration: 3.0 sec, Samples: 48000

🧪 Тестирование препроцессора на примере...


In [12]:
class AudioAugmentation:
    """Упрощенная аугментация аудио (БЕЗ librosa effects)"""
    
    def __init__(self, sr=16000, target_samples=48000):
        self.sr = sr
        self.target_samples = target_samples
    
    def add_noise(self, audio: np.ndarray, noise_level: float = 0.005) -> np.ndarray:
        """Добавляет белый шум"""
        noise = np.random.randn(len(audio)) * noise_level
        return audio + noise
    
    def volume_change(self, audio: np.ndarray, factor: float = 1.0) -> np.ndarray:
        """Изменяет громкость"""
        return audio * factor
    
    def polarity_inversion(self, audio: np.ndarray) -> np.ndarray:
        """Инвертирует полярность"""
        return -audio
    
    def augment(self, audio: np.ndarray, prob: float = 0.5) -> np.ndarray:
        """Применяет безопасную аугментацию"""
        if random.random() < prob:
            choice = random.choice(['noise', 'volume', 'polarity'])
            
            if choice == 'noise':
                audio = self.add_noise(audio, noise_level=random.uniform(0.001, 0.01))
            elif choice == 'volume':
                factor = random.uniform(0.8, 1.2)
                audio = self.volume_change(audio, factor=factor)
            elif choice == 'polarity':
                if random.random() < 0.5:
                    audio = self.polarity_inversion(audio)
        
        return audio

augmenter = AudioAugmentation(sr=16000, target_samples=48000)
print("✅ Упрощенная аугментация готова!")


✅ Упрощенная аугментация готова!


In [13]:
# ============= ИСПРАВЛЕННАЯ ЯЧЕЙКА 7: PYTORCH DATASET ДЛЯ PYARA (БЕЗ TYPE HINT) =============

# ⚠️ ДОБАВЛЯЕМ ИМПОРТЫ В НАЧАЛО!
from torch.utils.data import Dataset
import torch
import numpy as np

"""
ПРАВИЛЬНАЯ СТРУКТУРА:
/kaggle/input/pyara/final_dataset/Real/ - файлы напрямую
/kaggle/input/pyara/final_dataset/Fake/ - файлы напрямую
"""

class ASVspoofDatasetPyAra(Dataset):
    """PyTorch Dataset для PyAra датасета"""
    
    def __init__(self, csv_path: str, raw_dir: str, preprocessor, augment: bool = False):
        """
        Args:
            csv_path: путь к CSV с метаданными (train/val/test split)
            raw_dir: корневая директория PyAra (/kaggle/input/pyara)
            preprocessor: AudioPreprocessor для обработки аудио
        """
        # Загрузить метаданные
        self.df = pd.read_csv(csv_path)
        self.raw_dir = raw_dir
        self.preprocessor = preprocessor
        
        # Пути к директориям с аудио
        self.dataset_dir = os.path.join(raw_dir, 'final_dataset')
        self.real_dir = os.path.join(self.dataset_dir, 'Real')
        self.fake_dir = os.path.join(self.dataset_dir, 'Fake')
        
        # Создать кэш путей к файлам
        self.file_cache = {}

        self.augment = augment  # НОВОЕ
        
        if self.augment:
            self.augmenter = AudioAugmentation(sr=16000)
            print(f"  🎨 Data Augmentation ВКЛЮЧЕНА для этого набора")
        
        print(f"📂 Создание кэша путей к файлам...")
        missing_count = 0
        
        for idx, row in tqdm_notebook(self.df.iterrows(), total=len(self.df), desc="Кэширование"):
            filename = row['filename']
            class_name = row.get('class_name', 'unknown')  # 'real' или 'fake'
            found = False
            
            # Файлы лежат напрямую в Real/ или Fake/ (без подпапок!)
            if class_name == 'real':
                audio_path = os.path.join(self.real_dir, filename)
                if os.path.exists(audio_path):
                    self.file_cache[idx] = audio_path
                    found = True
            
            elif class_name == 'fake':
                audio_path = os.path.join(self.fake_dir, filename)
                if os.path.exists(audio_path):
                    self.file_cache[idx] = audio_path
                    found = True
            
            if not found:
                missing_count += 1
        
        # Результат кэширования
        found_count = len(self.file_cache)
        total_count = len(self.df)
        
        if missing_count > 0:
            print(f"\n⚠️ Найдено {found_count}/{total_count} файлов ({missing_count} отсутствуют)")
        else:
            print(f"\n✅ Все {found_count} файлов найдены!")
        
        # Статистика
        print(f"\n📊 Dataset статистика:")
        print(f"   Всего файлов: {len(self.df)}")
        print(f"   - Real: {(self.df['label'] == 0).sum()}")
        print(f"   - Fake: {(self.df['label'] == 1).sum()}")
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        """Возвращает (аудио_тензор, метка)"""
        row = self.df.iloc[idx]
        
        # Получить путь из кэша
        if idx in self.file_cache:
            audio_path = self.file_cache[idx]
            audio = self.preprocessor.preprocess(audio_path)
        else:
            # Если файл не найден - вернуть тишину
            audio = np.zeros(self.preprocessor.target_samples)
        
        if audio is None:
            audio = np.zeros(self.preprocessor.target_samples)

        audio = self.preprocessor.preprocess(audio_path)
        if self.augment and audio is not None:
            audio = self.augmenter.augment(audio, prob=0.3)
        
        # Метка класса
        label = int(row['label'])
        
        # Конвертировать в torch tensor
        audio_tensor = torch.FloatTensor(audio)
        
        return audio_tensor, label


# ===== СОЗДАТЬ DATASET'Ы =====
print("=" * 70)
print("🔨 СОЗДАНИЕ PYTORCH DATASET'ОВ ДЛЯ PYARA")
print("=" * 70)

train_dataset = ASVspoofDatasetPyAra(
    csv_path=f'{DATA_DIR}/processed/train_split.csv',
    raw_dir=raw_dir,
    preprocessor=preprocessor,
    augment=True  # ВКЛЮЧАЕМ для train
)

val_dataset = ASVspoofDatasetPyAra(
    csv_path=f'{DATA_DIR}/processed/val_split.csv',
    raw_dir=raw_dir,
    preprocessor=preprocessor,
    augment=False  # НЕ используем для val
)


print()

test_dataset = ASVspoofDatasetPyAra(
    csv_path=f'{DATA_DIR}/processed/test_split.csv',
    raw_dir=raw_dir,
    preprocessor=preprocessor
)

print(f"\n✅ Dataset'ы созданы:")
print(f"   Train: {len(train_dataset)} файлов")
print(f"   Val:   {len(val_dataset)} файлов")
print(f"   Test:  {len(test_dataset)} файлов")

# Тестировать получение одного образца
print(f"\n🧪 Тестирование получения образца:")
audio_sample, label_sample = train_dataset[0]
label_name = 'Real' if label_sample == 0 else 'Fake'
print(f"   Audio shape: {audio_sample.shape}")
print(f"   Label: {label_sample} ({label_name})")
print(f"   Audio stats: min={audio_sample.min():.3f}, max={audio_sample.max():.3f}, mean={audio_sample.mean():.3f}")

# КРИТИЧЕСКАЯ ПРОВЕРКА
if audio_sample.abs().sum() == 0:
    print(f"\n❌ КРИТИЧЕСКАЯ ОШИБКА: Аудио состоит из нулей!")
    print(f"   Файлы не загружаются правильно")
else:
    print(f"\n✅ Аудио загружено корректно (не нули)")

🔨 СОЗДАНИЕ PYTORCH DATASET'ОВ ДЛЯ PYARA
  🎨 Data Augmentation ВКЛЮЧЕНА для этого набора
📂 Создание кэша путей к файлам...


Кэширование:   0%|          | 0/98871 [00:00<?, ?it/s]


✅ Все 98871 файлов найдены!

📊 Dataset статистика:
   Всего файлов: 98871
   - Real: 36056
   - Fake: 62815
📂 Создание кэша путей к файлам...


Кэширование:   0%|          | 0/21187 [00:00<?, ?it/s]


✅ Все 21187 файлов найдены!

📊 Dataset статистика:
   Всего файлов: 21187
   - Real: 7726
   - Fake: 13461

📂 Создание кэша путей к файлам...


Кэширование:   0%|          | 0/21187 [00:00<?, ?it/s]


✅ Все 21187 файлов найдены!

📊 Dataset статистика:
   Всего файлов: 21187
   - Real: 7727
   - Fake: 13460

✅ Dataset'ы созданы:
   Train: 98871 файлов
   Val:   21187 файлов
   Test:  21187 файлов

🧪 Тестирование получения образца:
   Audio shape: torch.Size([48000])
   Label: 0 (Real)
   Audio stats: min=-0.970, max=0.825, mean=-0.000

✅ Аудио загружено корректно (не нули)


In [14]:
from torch.utils.data import WeightedRandomSampler

# Параметры
batch_size = 128  # Увеличил до 128 для ускорения обучения
num_workers = 2
pin_memory = True

print("="*70)
print("⚙️  СОЗДАНИЕ DATALOADERS (С БАЛАНСИРОВКОЙ)")
print("="*70)

# Создать веса для сэмплера (балансировка классов)
train_labels = train_dataset.df['label'].values
class_counts = np.bincount(train_labels)
class_weights = 1.0 / class_counts
sample_weights = class_weights[train_labels]

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

# TRAIN LOADER с сэмплером
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    sampler=sampler,  # Используем sampler вместо shuffle
    num_workers=num_workers,
    pin_memory=pin_memory,
    drop_last=True
)

print(f"\n✅ Train loader создан")
print(f"   Батчей: {len(train_loader)}")
print(f"   Batch size: {batch_size}")

# Проверить батчи
print(f"\n✅ Проверка батчей:")
for batch_idx, (audio, labels) in enumerate(train_loader):
    if batch_idx >= 5:
        break
    bonafide = (labels == 0).sum()
    spoofed = (labels == 1).sum()
    print(f"   Батч {batch_idx}: Bonafide={bonafide}, Spoofed={spoofed}")

# VAL LOADER
val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=pin_memory
)

print(f"\n✅ Val loader создан")
print(f"   Батчей: {len(val_loader)}")

print("\n" + "="*70)
print("✅ DATALOADERS ГОТОВЫ")
print("="*70)

⚙️  СОЗДАНИЕ DATALOADERS (С БАЛАНСИРОВКОЙ)

✅ Train loader создан
   Батчей: 772
   Batch size: 128

✅ Проверка батчей:
   Батч 0: Bonafide=62, Spoofed=66
   Батч 1: Bonafide=50, Spoofed=78
   Батч 2: Bonafide=70, Spoofed=58
   Батч 3: Bonafide=62, Spoofed=66
   Батч 4: Bonafide=54, Spoofed=74

✅ Val loader создан
   Батчей: 166

✅ DATALOADERS ГОТОВЫ


In [15]:
# ============= ДИАГНОСТИКА WAVLM =============

# Определить device, если ещё не определено
if 'device' not in dir():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"🔧 Устройство: {device}")
    
class WavLMFeatureExtractor(nn.Module):
    """Извлекатель признаков WavLM с правильной обработкой выхода"""

    def __init__(self, model_name: str = "microsoft/wavlm-large"):
        super().__init__()

        print(f"📥 Загрузка {model_name}...")
        self.model = AutoModel.from_pretrained(model_name)

        # КРИТИЧНО: Проверить выходную размерность
        config = self.model.config
        self.hidden_dim = config.hidden_size
        print(f"✅ WavLM Hidden Dimension: {self.hidden_dim}")

        # Заморозить параметры
        for param in self.model.parameters():
            param.requires_grad = False

        self.model.eval()
        print(f"✅ WavLM загружена и заморожена!")

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Входные данные: [batch_size, num_samples]
        Выходные данные: [batch_size, seq_len, hidden_dim]
        """
        with torch.no_grad():
            outputs = self.model(x, output_hidden_states=False)
            embeddings = outputs.last_hidden_state  # [B, T, hidden_dim]
        return embeddings

# Загрузить и проверить
print("="*70)
print("🔧 ДИАГНОСТИКА WAVLM")
print("="*70)

wavlm_extractor = WavLMFeatureExtractor(model_name="microsoft/wavlm-large")
wavlm_extractor = wavlm_extractor.to(device)

# ТЕСТИРОВАНИЕ - ОЧЕНЬ ВАЖНО!
print(f"\n🧪 ТЕСТИРОВАНИЕ WAVLM ВЫХОДА:")
with torch.no_grad():
    test_audio = torch.randn(2, 16000*3).to(device)  # [2, 48000]
    test_embeddings = wavlm_extractor(test_audio)

    print(f"   Input shape:  {test_audio.shape} (batch=2, samples=48000)")
    print(f"   Output shape: {test_embeddings.shape}")
    print(f"   Hidden dim:   {wavlm_extractor.hidden_dim}")

    # Получить размерность после pooling
    pooled = test_embeddings.mean(dim=1)  # [B, hidden_dim]
    print(f"   After pooling: {pooled.shape}")

# СОХРАНИТЬ РАЗМЕРНОСТЬ ДЛЯ МОДЕЛИ
wavlm_hidden_dim = wavlm_extractor.hidden_dim

🔧 ДИАГНОСТИКА WAVLM
📥 Загрузка microsoft/wavlm-large...


config.json: 0.00B [00:00, ?B/s]

2026-01-05 02:27:06.849910: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767580027.457146      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767580027.595348      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767580029.036541      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767580029.036571      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767580029.036574      55 computation_placer.cc:177] computation placer alr

pytorch_model.bin:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

✅ WavLM Hidden Dimension: 1024
✅ WavLM загружена и заморожена!

🧪 ТЕСТИРОВАНИЕ WAVLM ВЫХОДА:
   Input shape:  torch.Size([2, 48000]) (batch=2, samples=48000)
   Output shape: torch.Size([2, 149, 1024])
   Hidden dim:   1024
   After pooling: torch.Size([2, 1024])


In [16]:
# ============= AASIST КЛАССИФИКАТОР (ИСПРАВЛЕННЫЙ) =============

class AAsiSTClassifier(nn.Module):
    """AASIST классификатор с динамической входной размерностью"""

    def __init__(self, input_dim: int = None, hidden_dim: int = 512,
                 num_classes: int = 2, dropout: float = 0.3):
        super().__init__()

        # Если input_dim не задан, использовать 768 (по умолчанию для WavLM-large)
        if input_dim is None:
            input_dim = 1024

        print(f"🏗️  AASIST Classifier:")
        print(f"   Input dim: {input_dim}")
        print(f"   Hidden dim: {hidden_dim}")

        # Encoder
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.BatchNorm1d(hidden_dim),
            nn.Dropout(dropout),
            
            # Дополнительный слой
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.BatchNorm1d(hidden_dim),
            nn.Dropout(dropout)
        )

        # Attention mechanism
        self.attention = nn.MultiheadAttention(
            embed_dim=hidden_dim,
            num_heads=8,
            dropout=dropout,
            batch_first=True
        )
        
        # Classifier с residual connection
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, num_classes)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, hidden_dim]
        batch_size = x.size(0)
        
        # Encoder
        x = self.encoder(x)  # [B, hidden_dim]
        
        # Для attention нужен sequence dim
        x_seq = x.unsqueeze(1)  # [B, 1, hidden_dim]
        
        # Self-attention
        attn_out, _ = self.attention(x_seq, x_seq, x_seq)  # [B, 1, hidden_dim]
        x_attn = attn_out.squeeze(1)  # [B, hidden_dim]
        
        # Residual connection
        x = x + x_attn  # [B, hidden_dim]
        
        # Classifier
        logits = self.classifier(x)  # [B, 2]
        
        return logits

# Создать модель с ПРАВИЛЬНОЙ размерностью
print("="*70)
print("🏗️  СОЗДАНИЕ AASIST КЛАССИФИКАТОРА")
print("="*70)

aasist_model = AAsiSTClassifier(
    input_dim=wavlm_hidden_dim,  # ИСПОЛЬЗУЕМ РЕАЛЬНУЮ РАЗМЕРНОСТЬ ИЗ WAVLM!
    hidden_dim=512,
    num_classes=2,
    dropout=0.3
)
aasist_model = aasist_model.to(device)

print("✅ AASIST модель создана!")
print(f"\n📊 Архитектура модели:")
print(aasist_model)

total_params = sum(p.numel() for p in aasist_model.parameters())
trainable_params = sum(p.numel() for p in aasist_model.parameters() if p.requires_grad)
print(f"\n📈 Параметры модели:")
print(f"   Всего: {total_params:,}")
print(f"   Обучаемых: {trainable_params:,}")

🏗️  СОЗДАНИЕ AASIST КЛАССИФИКАТОРА
🏗️  AASIST Classifier:
   Input dim: 1024
   Hidden dim: 512
✅ AASIST модель создана!

📊 Архитектура модели:
AAsiSTClassifier(
  (encoder): Sequential(
    (0): Linear(in_features=1024, out_features=512, bias=True)
    (1): ReLU()
    (2): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=512, out_features=512, bias=True)
    (5): ReLU()
    (6): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (7): Dropout(p=0.3, inplace=False)
  )
  (attention): MultiheadAttention(
    (out_proj): NonDynamicallyQuantizableLinear(in_features=512, out_features=512, bias=True)
  )
  (classifier): Sequential(
    (0): Linear(in_features=512, out_features=256, bias=True)
    (1): ReLU()
    (2): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_feature

In [17]:
"""
УПРАВЛЕНИЕ CHECKPOINTS

Функции для сохранения и загрузки лучших моделей
"""

def save_checkpoint(model: nn.Module,
                   optimizer,
                   epoch: int,
                   loss: float,
                   save_path: str):
    """Сохраняет checkpoint модели"""

    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': loss,
        'timestamp': datetime.now().isoformat()
    }

    torch.save(checkpoint, save_path)
    print(f"✅ Checkpoint сохранён: {os.path.basename(save_path)}")

def load_checkpoint(model: nn.Module,
                   optimizer,
                   checkpoint_path: str,
                   device):
    """Загружает checkpoint модели"""

    if not os.path.exists(checkpoint_path):
        print(f"❌ Checkpoint не найден: {checkpoint_path}")
        return 0, float('inf')

    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

    epoch = checkpoint['epoch']
    loss = checkpoint['loss']

    print(f"✅ Checkpoint загружен: epoch {epoch}, loss {loss:.4f}")
    return epoch, loss

# Инициализировать директорию для checkpoints
checkpoint_dir = f'{MODELS_DIR}/checkpoints'
os.makedirs(checkpoint_dir, exist_ok=True)

print("✅ Checkpoint management готов!")
print(f"   Dir: {checkpoint_dir}")

✅ Checkpoint management готов!
   Dir: /kaggle/working/models/checkpoints


In [18]:
"""
МЕТРИКИ ДЛЯ ANTI-SPOOFING

- Accuracy, Precision, Recall, F1
- FAR (False Accept Rate), FRR (False Reject Rate)
- EER (Equal Error Rate), AUC-ROC
"""

def compute_metrics(y_true: np.ndarray,
                   y_pred: np.ndarray,
                   y_score: np.ndarray) -> dict:
    """Вычисляет метрики для anti-spoofing"""

    # ✅ КЛЮЧЕВОЕ ИСПРАВЛЕНИЕ: labels=[0, 1]
    # Это ГАРАНТИРУЕТ размер (2, 2) даже если одного класса нет!
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    far = fp / (fp + tn) if (fp + tn) > 0 else 0
    frr = fn / (fn + tp) if (fn + tp) > 0 else 0

    # Защита от edge case'ов
    try:
        auc = roc_auc_score(y_true, y_score)
    except ValueError:
        auc = 0.0

    try:
        fpr, fnr, thresholds = roc_curve(y_true, y_score)
        abs_diffs = np.abs(fpr - fnr)
        eer_idx = np.argmin(abs_diffs)
        eer = fpr[eer_idx]
    except (ValueError, IndexError):
        eer = 0.5

    return {
        'TP': tp, 'FP': fp, 'FN': fn, 'TN': tn,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1': f1,
        'FAR': far,
        'FRR': frr,
        'EER': eer,
        'AUC': auc
    }

print("✅ compute_metrics (исправленная) готова!")

✅ compute_metrics (исправленная) готова!


In [21]:
# ============= ОБУЧАЮЩИЙ ЦИКЛ (ИСПРАВЛЕННЫЙ) =============

from torch.amp import autocast, GradScaler

use_amp = True
scaler = GradScaler('cuda') if use_amp else None 

print(f"✅ Mixed Precision (FP16): {'ВКЛЮЧЕНО' if use_amp else 'ВЫКЛЮЧЕНО'}")

def train_epoch(model: nn.Module,
               wavlm: nn.Module,
               train_loader: DataLoader,
               optimizer,
               criterion,
               device,
               epoch: int,
               use_amp=False,
               scaler=None) -> tuple:
    """Один цикл обучения"""

    model.train()
    total_loss = 0
    all_preds = []
    all_labels = []

    pbar = tqdm_notebook(train_loader, desc=f"Train Epoch {epoch+1}")

    for batch_idx, (audio, labels) in enumerate(pbar):
        audio = audio.to(device)
        labels = labels.to(device)

        # Forward pass: WavLM
        with torch.no_grad():
            wavlm_embeddings = wavlm(audio)  # [B, T, hidden_dim]
            # КРИТИЧНО: Global Average Pooling по временной оси
            wavlm_pooled = wavlm_embeddings.mean(dim=1)  # [B, hidden_dim]

        if use_amp:
            with autocast():
                logits = model(wavlm_pooled)
                loss = criterion(logits, labels)
        else:
            logits = model(wavlm_pooled)
            loss = criterion(logits, labels)
        
        # Backward с GradScaler
        optimizer.zero_grad()

        if use_amp:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        # Metrics
        total_loss += loss.item()
        preds = logits.argmax(dim=1).cpu().detach().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().detach().numpy())

        pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    avg_loss = total_loss / len(train_loader)
    metrics = compute_metrics(
        np.array(all_labels),
        np.array(all_preds),
        np.array(all_preds)
    )

    return avg_loss, metrics

def validate(model: nn.Module,
            wavlm: nn.Module,
            val_loader: DataLoader,
            criterion,
            device) -> tuple:
    """Валидация модели"""

    model.eval()
    total_loss = 0
    all_preds = []
    all_scores = []
    all_labels = []

    with torch.no_grad():
        pbar = tqdm_notebook(val_loader, desc="Validation")
        for audio, labels in pbar:
            audio = audio.to(device)
            labels = labels.to(device)

            # Forward pass: WavLM
            wavlm_embeddings = wavlm(audio)  # [B, T, hidden_dim]
            # КРИТИЧНО: Global Average Pooling
            wavlm_pooled = wavlm_embeddings.mean(dim=1)  # [B, hidden_dim]
            logits = model(wavlm_pooled)

            # Loss
            loss = criterion(logits, labels)
            total_loss += loss.item()

            # Metrics
            preds = logits.argmax(dim=1).cpu().numpy()
            scores = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()

            all_preds.extend(preds)
            all_scores.extend(scores)
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(val_loader)
    metrics = compute_metrics(
        np.array(all_labels),
        np.array(all_preds),
        np.array(all_scores)
    )

    return avg_loss, metrics

print("✅ Функции обучения готовы!")

✅ Mixed Precision (FP16): ВКЛЮЧЕНО
✅ Функции обучения готовы!


In [1]:
# ============= ЯЧЕЙКА 14: БЕЗОПАСНОЕ ОБУЧЕНИЕ (ИСПРАВЛЕННАЯ) =============

# Загрузить веса классов
class_weights_dict = None
class_weights_path = f'{DATA_DIR}/processed/class_weights.json'
if os.path.exists(class_weights_path):
    import json
    with open(class_weights_path, 'r') as f:
        class_weights_dict = json.load(f)
    print(f"✅ Веса классов загружены")
else:
    print(f"⚠️  Веса не найдены, используются default")

# Гиперпараметры
learning_rate = 5e-5  # ИСПРАВЛЕНИЕ: уменьшено с 1e-3
num_epochs = 10
patience = 7

# Оптимайзер
optimizer = optim.Adam(aasist_model.parameters(),
                       lr=learning_rate,
                       weight_decay=1e-5)

optimizer = optim.Adam(aasist_model.parameters(), lr=learning_rate, weight_decay=1e-5)

# НОВОЕ: Добавляем scheduler
from torch.optim.lr_scheduler import ReduceLROnPlateau

scheduler = ReduceLROnPlateau(
    optimizer, 
    mode='min', 
    factor=0.5,      # Уменьшаем LR в 2 раза
    patience=3,       # После 3 эпох без улучшения
    min_lr=1e-7
)

# Loss function с весами
criterion = nn.CrossEntropyLoss()
print(f"⚠️  Используется обычная loss")

# КРИТИЧЕСКАЯ ДИАГНОСТИКА
print("\n" + "="*70)
print("🔍 ДИАГНОСТИКА ПЕРЕД ОБУЧЕНИЕМ")
print("="*70)

audio_test, labels_test = next(iter(train_loader))
audio_test = audio_test.to(device)
labels_test = labels_test.to(device)

with torch.no_grad():
    wavlm_out = wavlm_extractor(audio_test)
    wavlm_pooled = wavlm_out.mean(dim=1)
    logits = aasist_model(wavlm_pooled)
    loss_test = criterion(logits, labels_test)
    probs = torch.softmax(logits, dim=1)

    print(f"\n📊 Тестовый батч:")
    print(f"   Logits: min={logits.min():.3f}, max={logits.max():.3f}, mean={logits.mean():.3f}")
    print(f"   Probabilities: min={probs.min():.3f}, max={probs.max():.3f}")
    print(f"   Loss: {loss_test:.6f}")
    print(f"   Labels в батче: {labels_test.unique()}")

    # Проверка на проблемы
    if loss_test == 0:
        print("\n❌ КРИТИЧЕСКАЯ ОШИБКА: Loss = 0!")
        print("   → Проверьте веса классов")
        print("   → Может быть, все примеры одного класса в батче")
    elif torch.isnan(loss_test):
        print("\n❌ КРИТИЧЕСКАЯ ОШИБКА: Loss = NaN!")
        print("   → Возможен градиентный взрыв")
        print("   → Уменьшите learning rate еще больше")
    elif loss_test > 100:
        print("\n⚠️  ПРЕДУПРЕЖДЕНИЕ: Loss слишком большой")
        print("   → Уменьшите learning rate")
    else:
        print("\n✅ Loss выглядит нормально, можно начинать обучение")

print("\n" + "="*70)

# История обучения
history = {
    'train_loss': [],
    'val_loss': [],
    'val_acc': [],
    'val_f1': [],
    'val_auc': [],
    'val_eer': []
}

best_val_loss = float('inf')
patience_counter = 0

print("="*70)
print("🚀 НАЧАЛО ОБУЧЕНИЯ")
print("="*70)
print(f"Learning rate: {learning_rate}")
print(f"Batch size: {batch_size}")
print(f"Max epochs: {num_epochs}")
print("="*70 + "\n")

# Обучение
for epoch in range(num_epochs):

    # Train
    train_loss, train_metrics = train_epoch(
        aasist_model, wavlm_extractor, train_loader,
        optimizer, criterion, device, epoch
    )

    # Validate
    val_loss, val_metrics = validate(
        aasist_model, wavlm_extractor, val_loader,
        criterion, device
    )

    scheduler.step(val_loss)

    # История
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_metrics['Accuracy'])
    history['val_f1'].append(val_metrics['F1'])
    history['val_auc'].append(val_metrics['AUC'])
    history['val_eer'].append(val_metrics['EER'])

    # Вывод
    print(f"Epoch {epoch+1:3d}/{num_epochs} | Train: {train_loss:.4f} | Val: {val_loss:.4f} | Acc: {val_metrics['Accuracy']:.3f} | F1: {val_metrics['F1']:.3f}")

    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        checkpoint_path = f'{checkpoint_dir}/module1_aasist_best.pth'
        save_checkpoint(aasist_model, optimizer, epoch, val_loss, checkpoint_path)
        print(f"   ✅ Новая лучшая модель!")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"\n⏹️  Early stopping на эпохе {epoch+1}")
            break

print("\n✅ ОБУЧЕНИЕ ЗАВЕРШЕНО")
print(f"Лучший val_loss: {best_val_loss:.4f}")

NameError: name 'DATA_DIR' is not defined

In [ ]:
# ============= ДИАГНОСТИКА МОДЕЛИ И ДАННЫХ =============

print("🔍 ПОЛНАЯ ДИАГНОСТИКА")
print("="*70)

# 1. Проверить батчи
print("\n1️⃣  БАТЧИ ДАННЫХ:")
audio_batch, labels_batch = next(iter(train_loader))
print(f"   Audio shape: {audio_batch.shape}")
print(f"   Labels shape: {labels_batch.shape}")
print(f"   Labels в батче: {labels_batch.unique()}")
print(f"   Bonafide в батче: {(labels_batch==0).sum()}")
print(f"   Spoofed в батче: {(labels_batch==1).sum()}")

# 2. Проверить WavLM output
print("\n2️⃣  WAVLM FEATURE EXTRACTOR:")
audio_batch = audio_batch.to(device)
with torch.no_grad():
    wavlm_out = wavlm_extractor(audio_batch)
    print(f"   WavLM output shape: {wavlm_out.shape}")
    print(f"   WavLM output range: [{wavlm_out.min():.3f}, {wavlm_out.max():.3f}]")
    print(f"   WavLM output mean: {wavlm_out.mean():.3f}")
    print(f"   WavLM has NaN: {torch.isnan(wavlm_out).any()}")
    print(f"   WavLM has Inf: {torch.isinf(wavlm_out).any()}")

# 3. Проверить pooling
print("\n3️⃣  POOLING:")
with torch.no_grad():
    wavlm_out = wavlm_extractor(audio_batch)
    pooled = wavlm_out.mean(dim=1)
    print(f"   Pooled shape: {pooled.shape}")
    print(f"   Pooled range: [{pooled.min():.3f}, {pooled.max():.3f}]")
    print(f"   Pooled has NaN: {torch.isnan(pooled).any()}")

# 4. Проверить AASIST модель
print("\n4️⃣  AASIST МОДЕЛЬ:")
with torch.no_grad():
    wavlm_out = wavlm_extractor(audio_batch)
    pooled = wavlm_out.mean(dim=1)
    logits = aasist_model(pooled)
    print(f"   Logits shape: {logits.shape}")
    print(f"   Logits range: [{logits.min():.3f}, {logits.max():.3f}]")
    print(f"   Logits has NaN: {torch.isnan(logits).any()}")
    print(f"   Logits has Inf: {torch.isinf(logits).any()}")

    # Проверить output модели
    probs = torch.softmax(logits, dim=1)
    preds = logits.argmax(dim=1)
    print(f"   Predictions: {preds}")
    print(f"   Probs range: [{probs.min():.3f}, {probs.max():.3f}]")

# 5. Проверить loss
print("\n5️⃣  LOSS FUNCTION:")
labels_batch = labels_batch.to(device)
with torch.no_grad():
    wavlm_out = wavlm_extractor(audio_batch)
    pooled = wavlm_out.mean(dim=1)
    logits = aasist_model(pooled)
    loss = criterion(logits, labels_batch)

    print(f"   Loss: {loss:.6f}")
    print(f"   Loss is NaN: {torch.isnan(loss)}")
    print(f"   Loss is Inf: {torch.isinf(loss)}")

# 6. Градиенты
print("\n6️⃣  ГРАДИЕНТЫ:")
wavlm_out = wavlm_extractor(audio_batch)
pooled = wavlm_out.mean(dim=1)
logits = aasist_model(pooled)
loss = criterion(logits, labels_batch)
loss.backward()

# Проверить градиенты модели
for name, param in aasist_model.named_parameters():
    if param.grad is not None:
        grad_norm = param.grad.norm()
        print(f"   {name}: grad_norm={grad_norm:.6f}")
        if grad_norm > 100:
            print(f"      ⚠️  ОГРОМНЫЙ ГРАДИЕНТ!")

print("\n" + "="*70)

In [ ]:
"""
ГРАФИКИ РЕЗУЛЬТАТОВ ОБУЧЕНИЯ

Loss, Accuracy, AUC, EER
"""

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Loss
axes[0, 0].plot(history['train_loss'], label='Train', marker='o', markersize=4, linewidth=2)
axes[0, 0].plot(history['val_loss'], label='Validation', marker='s', markersize=4, linewidth=2)
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].set_title('Training vs Validation Loss')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Accuracy
axes[0, 1].plot(history['val_acc'], label='Val Accuracy', marker='o', markersize=4, linewidth=2, color='green')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Accuracy')
axes[0, 1].set_title('Validation Accuracy')
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].legend()

# AUC-ROC
axes[1, 0].plot(history['val_auc'], label='Val AUC', marker='o', markersize=4, linewidth=2, color='orange')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('AUC')
axes[1, 0].set_title('Validation AUC-ROC')
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].legend()

# EER
axes[1, 1].plot(history['val_eer'], label='Val EER', marker='o', markersize=4, linewidth=2, color='red')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('EER')
axes[1, 1].set_title('Equal Error Rate (ниже лучше)')
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].legend()

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/plots/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Графики сохранены в plots/")

# Сохранить историю как JSON
with open(f'{RESULTS_DIR}/metrics/training_history.json', 'w') as f:
    json.dump(history, f, indent=2)

print("✅ История обучения сохранена в metrics/")

In [ ]:
# ============= ТЕСТИРОВАНИЕ (ИСПРАВЛЕННОЕ) =============

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("\n" + "="*70)
print("🧪 ТЕСТИРОВАНИЕ НА TEST НАБОРЕ")
print("="*70)

aasist_model.eval()
all_preds = []
all_scores = []
all_labels = []

with torch.no_grad():
    for audio, labels in tqdm_notebook(test_loader, desc="Testing"):
        audio = audio.to(device)
        labels = labels.to(device)

        # Forward pass
        wavlm_embeddings = wavlm_extractor(audio)
        wavlm_pooled = wavlm_embeddings.mean(dim=1)  # POOLING ОБЯЗАТЕЛЕН
        logits = aasist_model(wavlm_pooled)

        # Predictions
        preds = logits.argmax(dim=1).cpu().numpy()
        scores = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()

        all_preds.extend(preds)
        all_scores.extend(scores)
        all_labels.extend(labels.cpu().numpy())

test_metrics = compute_metrics(
    np.array(all_labels),
    np.array(all_preds),
    np.array(all_scores)
)

print("\n" + "="*70)
print("📊 РЕЗУЛЬТАТЫ ТЕСТИРОВАНИЯ")
print("="*70)
for metric in ['Accuracy', 'Precision', 'Recall', 'F1', 'AUC', 'EER', 'FAR', 'FRR']:
    value = test_metrics[metric]
    print(f"{metric:12s}: {value:.4f}")

In [ ]:
# ============= ЯЧЕЙКА 17: ROC-CURVE И CONFUSION MATRIX (ИСПРАВЛЕННАЯ) =============

from sklearn.metrics import ConfusionMatrixDisplay

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC Curve
fpr, tpr, thresholds = roc_curve(all_labels, all_scores)
roc_auc = auc(fpr, tpr)

axes[0].plot(fpr, tpr, color='darkorange', lw=2.5,
             label=f'ROC Curve (AUC = {roc_auc:.4f})')
axes[0].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--',
             label='Random Classifier')
axes[0].set_xlim([0.0, 1.0])
axes[0].set_ylim([0.0, 1.05])
axes[0].set_xlabel('False Positive Rate (FAR)', fontsize=11)
axes[0].set_ylabel('True Positive Rate (TPR)', fontsize=11)
axes[0].set_title('ROC Curve - AASIST Anti-Spoofing', fontsize=12, fontweight='bold')
axes[0].legend(loc="lower right", fontsize=10)
axes[0].grid(True, alpha=0.3)

# Confusion Matrix - ИСПРАВЛЕННАЯ
# КРИТИЧНО: labels=[0, 1] гарантирует размер (2, 2)!
cm = confusion_matrix(all_labels, all_preds, labels=[0, 1])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Bonafide', 'Spoofed'])
disp.plot(ax=axes[1], cmap='Blues', values_format='d')
axes[1].set_title('Confusion Matrix - AASIST Anti-Spoofing', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/plots/module1_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Графики оценки сохранены!")

In [ ]:
"""
ИТОГОВЫЙ ОТЧЁТ МОДУЛЯ 1
"""

print("\n" + "="*80)
print("📋 ИТОГОВЫЙ ОТЧЁТ - МОДУЛЬ 1 (AASIST + WavLM)")
print("="*80)

print(f"\n🏗️  АРХИТЕКТУРА МОДЕЛИ:")
print(f"   Feature Extractor: WavLM-large (frozen, 768-dim)")
print(f"   Classifier: AASIST")
print(f"   - Encoder: Linear(768→256) + ReLU + BatchNorm + Dropout")
print(f"   - Attention: MultiheadAttention (8 heads)")
print(f"   - Pooling: Global Average Pooling")
print(f"   - Classifier: Linear(256→128) + Linear(128→2)")

print(f"\n📊 ПАРАМЕТРЫ МОДЕЛИ:")
print(f"   Всего параметров: {total_params:,}")
print(f"   Обучаемых: {trainable_params:,}")
print(f"   WavLM (frozen): {sum(p.numel() for p in wavlm_extractor.parameters()):,}")

print(f"\n📈 ДАННЫЕ ОБУЧЕНИЯ:")
print(f"   Train: {len(train_dataset):5d} файлов")
print(f"   Val:   {len(val_dataset):5d} файлов")
print(f"   Test:  {len(test_dataset):5d} файлов")
print(f"   Batch size: {batch_size}")

print(f"\n🎓 РЕЗУЛЬТАТЫ ОБУЧЕНИЯ:")
print(f"   Epochs completed: {epoch + 1}")
print(f"   Best val loss: {min(history['val_loss']):.4f}")
print(f"   Best val AUC: {max(history['val_auc']):.4f}")
print(f"   Min val EER: {min(history['val_eer']):.4f}")

print(f"\n🎯 ТЕСТОВЫЕ МЕТРИКИ:")
print(f"   Accuracy:  {test_metrics['Accuracy']:.4f}")
print(f"   Precision: {test_metrics['Precision']:.4f}")
print(f"   Recall:    {test_metrics['Recall']:.4f}")
print(f"   F1-score:  {test_metrics['F1']:.4f}")
print(f"   AUC-ROC:   {test_metrics['AUC']:.4f}")
print(f"   EER:       {test_metrics['EER']:.4f}")
print(f"   FAR:       {test_metrics['FAR']:.4f}")
print(f"   FRR:       {test_metrics['FRR']:.4f}")

print(f"\n💾 СОХРАНЁННЫЕ ФАЙЛЫ:")
print(f"   ✅ Модель: {checkpoint_dir}/module1_aasist_best.pth")
print(f"   ✅ История: {RESULTS_DIR}/metrics/training_history.json")
print(f"   ✅ Результаты: {RESULTS_DIR}/metrics/module1_test_results.csv")
print(f"   ✅ Графики: {RESULTS_DIR}/plots/")

print("\n" + "="*80)
print("✅ МОДУЛЬ 1 УСПЕШНО ЗАВЕРШЁН!")
print("="*80)

In [ ]:
# ============================================================
# СОХРАНЕНИЕ МОДЕЛИ ПОСЛЕ ПРЕРЫВАНИЯ
# ============================================================

import os
import torch

print("=" * 70)
print("💾 ВОССТАНОВЛЕНИЕ И СОХРАНЕНИЕ МОДЕЛИ")
print("=" * 70)

# Пути
CHECKPOINT_DIR = "/kaggle/working/models/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, "module1_aasist_best.pth")

print(f"\n🔍 Поиск существующих checkpoint'ов...")

# 1️⃣ ПРОВЕРИТЬ: может checkpoint уже есть в памяти kernel'а?
if os.path.exists(CHECKPOINT_PATH):
    print(f"✅ Checkpoint уже существует: {CHECKPOINT_PATH}")
    print(f"   Размер: {os.path.getsize(CHECKPOINT_PATH) / 1024:.1f} KB")
else:
    print(f"⚠️  Checkpoint не найден в {CHECKPOINT_PATH}")
    
    # 2️⃣ ЕСЛИ МОДЕЛЬ ВСЕ ЕЩЕ В ПАМЯТИ - СОХРАНИ ЕЁ
    # (это работает только если ты не перезагружал kernel после обучения)
    
    if 'aasist_model' in dir():
        print(f"\n✅ Модель найдена в памяти kernel'а!")
        print(f"   Сохраняю...")
        
        checkpoint = {
            'model_state_dict': aasist_model.state_dict(),
            'model_architecture': 'AAiST',
            'wavlm_hidden_dim': 1024,
            'hidden_dim': 256,
            'num_classes': 2,
        }
        
        torch.save(checkpoint, CHECKPOINT_PATH)
        print(f"   ✅ Сохранено в {CHECKPOINT_PATH}")
        print(f"   Размер: {os.path.getsize(CHECKPOINT_PATH) / 1024:.1f} KB")
    else:
        print(f"\n❌ Модель НЕ найдена в памяти")
        print(f"   Нужно ПЕРЕСТАРТОВАТЬ обучение или использовать РЕШЕНИЕ 2")

print("\n" + "=" * 70)


In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModel
import numpy as np
import librosa
import os
import pickle
from tqdm.notebook import tqdm

class DeepfakeInferenceModule:
    """
    Инференсный модуль для обучения Fusion.
    
    Применяет обученную WavLM + AAiST к аудиофайлам и возвращает вероятность deepfake.
    """
    
    def __init__(self, 
                 checkpoint_path: str,
                 device: str = "cuda"):
        """
        Args:
            checkpoint_path: путь к /kaggle/working/models/checkpoints/module1_aasist_best.pth
            device: "cuda" или "cpu"
        """
        
        self.device = torch.device(device)
        self.sr = 16000
        self.target_duration = 3.0
        self.target_samples = int(self.sr * self.target_duration)  # 48000
        
        print("🔧 DeepfakeInferenceModule инициализация...")
        
        # 1️⃣ ЗАГРУЗИТЬ WAVLM
        print(" 📥 Загрузка WavLM...")
        self.wavlm = AutoModel.from_pretrained("microsoft/wavlm-large")
        self.wavlm.eval()
        for param in self.wavlm.parameters():
            param.requires_grad = False
        self.wavlm = self.wavlm.to(self.device)
        self.wavlm_hidden_dim = self.wavlm.config.hidden_size  # 1024
        print(f" ✅ WavLM загружена (hidden_dim={self.wavlm_hidden_dim})")
        
        # 2️⃣ СОЗДАТЬ И ЗАГРУЗИТЬ AASIST
        print(" 🏗️ Загрузка AAiST классификатора...")
        self.aasist = self._create_aasist_model()
        
        # Загрузить checkpoint
        if not os.path.exists(checkpoint_path):
            raise FileNotFoundError(f"Checkpoint не найден: {checkpoint_path}")
        
        checkpoint = torch.load(checkpoint_path, map_location=self.device)
        self.aasist.load_state_dict(checkpoint['model_state_dict'])
        self.aasist.eval()
        print(f" ✅ AAiST загружена из checkpoint")
        
        print("✅ DeepfakeInferenceModule готов к инференсу!")
    
    def _create_aasist_model(self):
        """Создаёт AAiST модель (ТОЧНО как при обучении)"""
        
        class AAiSTClassifier(nn.Module):
            def __init__(self, input_dim: int = 1024, hidden_dim: int = 256):
                super().__init__()
                
                self.encoder = nn.Sequential(
                    nn.Linear(input_dim, hidden_dim),
                    nn.ReLU(),
                    nn.BatchNorm1d(hidden_dim),
                    nn.Dropout(0.2)
                )
                
                self.classifier = nn.Sequential(
                    nn.Linear(hidden_dim, hidden_dim // 2),
                    nn.ReLU(),
                    nn.Dropout(0.2),
                    nn.Linear(hidden_dim // 2, 2)
                )
            
            def forward(self, x):
                x = self.encoder(x)
                logits = self.classifier(x)
                return logits
        
        model = AAiSTClassifier(input_dim=self.wavlm_hidden_dim)
        model = model.to(self.device)
        return model
    
    def _load_audio(self, audio_path: str) -> np.ndarray:
        """Загружает и нормализует аудио"""
        try:
            y, sr = librosa.load(audio_path, sr=self.sr, mono=True)
            
            # Нормализация
            max_val = np.max(np.abs(y))
            if max_val > 0:
                y = y / (max_val + 1e-7)
            
            # Фильтр высоких частот
            from scipy import signal
            sos = signal.butter(5, 200, 'hp', fs=self.sr, output='sos')
            y = signal.sosfilt(sos, y)
            
            # Pad/Trim до целевой длины
            if len(y) > self.target_samples:
                y = y[:self.target_samples]
            else:
                y = np.pad(y, (0, self.target_samples - len(y)), mode='constant')
            
            return y
        
        except Exception as e:
            return None
    
    def get_score(self, audio_path: str) -> float:
        """
        Получает вероятность deepfake для одного файла.
        
        Args:
            audio_path: путь к аудиофайлу
        
        Returns:
            float: вероятность что это deepfake [0, 1]
        """
        
        try:
            # Загрузить аудио
            y = self._load_audio(audio_path)
            if y is None:
                return 0.5  # Fallback
            
            # Преобразовать в тензор
            audio_tensor = torch.FloatTensor(y).unsqueeze(0).to(self.device)  # [1, 48000]
            
            # Извлечь признаки (WavLM)
            with torch.no_grad():
                wavlm_embeddings = self.wavlm(audio_tensor)
                # Global Average Pooling по временной оси
                wavlm_pooled = wavlm_embeddings.last_hidden_state.mean(dim=1)  # [1, 1024]
            
            # Классификация (AAiST)
            with torch.no_grad():
                logits = self.aasist(wavlm_pooled)  # [1, 2]
                probs = torch.softmax(logits, dim=1)  # [1, 2]
            
            # Вероятность deepfake (класс 1)
            prob_deepfake = float(probs[0, 1].cpu().numpy())
            
            return prob_deepfake
        
        except Exception as e:
            print(f"  ⚠️  Ошибка на {os.path.basename(audio_path)}: {e}")
            return 0.5
    
    def get_scores(self, audio_paths: list) -> list:
        """Получает scores для списка файлов"""
        scores = []
        for audio_path in audio_paths:
            score = self.get_score(audio_path)
            scores.append(score)
        return scores
